# Email Campaign Performance Data Cleaning
Cleaning a ~1,000-row SaaS email campaign dataset to enable analysis of open/click rates.
**Dataset**: Tracks email sends, opens, clicks, conversions, and unsubscribes.
**Columns**: EmailID, RecipientEmail, CampaignID, SendTime, OpenTime, ClickTime, Conversion, Unsubscribe.
**Tasks**:
1. Unify dates to 'YYYY-MM-DD HH:MM:SS', fixing invalid formats.
2. Validate RecipientEmail.
3. Standardize CampaignID.
4. Clean Conversion.
5. Resolve duplicates.
6. Fill blanks.

In [76]:
# Imports and Configuration
import pandas as pd
import numpy as np
import re

# Set random seed for reproducibility
np.random.seed(42)

# Configure pandas for future compatibility
pd.set_option('future.no_silent_downcasting', True)

# Load dataset
df = pd.read_csv('email_campaign.csv')
df_clean = df.copy()  # Preserve original for reference

# Initial overview
print("Initial Data Overview:")
print(df_clean.info())

print("\nSample Data:")
print(df_clean.head())

# Save original for reproducibility
df.to_csv('original_email_campaign.csv', index=False)

Initial Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   EmailID         1020 non-null   object
 1   RecipientEmail  998 non-null    object
 2   CampaignID      1020 non-null   object
 3   SendTime        1020 non-null   object
 4   OpenTime        305 non-null    object
 5   ClickTime       162 non-null    object
 6   Conversion      802 non-null    object
 7   Unsubscribe     1020 non-null   object
dtypes: object(8)
memory usage: 63.9+ KB
None

Sample Data:
     EmailID      RecipientEmail CampaignID             SendTime  \
0  EMAIL1001    user16@gmail.com    CAMP103  01/02/2024 18:33:49   
1  EMAIL1002   user192@gmail.com    camp102  01/07/2024 00:23:25   
2  EMAIL1003   user177@yahoo.com    CAMP101  01/05/2024 05:23:45   
3  EMAIL1004  user89@outlook.com    CAMP102  2024-01-12 04:25:51   
4  EMAIL1005    user60@yahoo.co

## Task 1: Unify Dates
Convert `SendTime`, `OpenTime`, `ClickTime` to 'YYYY-MM-DD HH:MM:SS', fixing invalid US format dates (e.g., '2024-13-01 10:00:00' as '01/13/2024'). Preserve `NaN` in `OpenTime`/`ClickTime` (~30% missing) as valid (no action taken).

In [78]:
# Task 1: Unify Dates

# Preserve original timestamps
for col in ['SendTime', 'OpenTime', 'ClickTime']:
    df_clean[f'Original{col}'] = df_clean[col]

# Function to normalize date formats
def normalize_date(timestamp):
    """Normalize MM/DD/YYYY or YYYY-MM-DD to YYYY-MM-DD HH:MM:SS, fix invalid months."""
    if pd.isna(timestamp):
        return timestamp
    # Try MM/DD/YYYY HH:MM:SS
    match_us = re.match(r'(\d{2})/(\d{2})/(\d{4})\s(\d{2}:\d{2}:\d{2})', timestamp)
    if match_us:
        month, day, year, time = match_us.groups()
        return f'{year}-{month}-{day} {time}'
    # Try YYYY-MM-DD HH:MM:SS with invalid month
    match_invalid = re.match(r'(\d{4})-(\d{2})-(\d{2})\s(\d{2}:\d{2}:\d{2})', timestamp)
    if match_invalid:
        year, month, day, time = match_invalid.groups()
        if int(month) > 12:  # Fix US format (e.g., '2024-13-01' → '2024-01-13')
            return f'{year}-{day}-{month} {time}'
        return timestamp  # Already YYYY-MM-DD
    return timestamp  # Unmatched, will become NaT

# Apply normalization and standardize
date_cols = ['SendTime', 'OpenTime', 'ClickTime']
for col in date_cols:
    df_clean[col] = df_clean[col].apply(normalize_date)
    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce', format='%Y-%m-%d %H:%M:%S')
    df_clean[col] = df_clean[col].dt.strftime('%Y-%m-%d %H:%M:%S').where(df_clean[col].notna())

In [79]:
# Verify Task 1
print("Task 1 Verification:")
pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'
for col in date_cols:
    valid = df_clean[col].str.match(pattern, na=False) | df_clean[col].isna()
    print(f"{col} invalid formats: {(~valid).sum()}")
    if (~valid).any():
        print(f"Invalid examples:\n{df_clean[~valid][[col]].head()}")
    parsed = pd.to_datetime(df_clean[col], errors='coerce', format='%Y-%m-%d %H:%M:%S')
    invalid_dates = parsed.isna() & df_clean[col].notna()
    print(f"{col} invalid dates: {invalid_dates.sum()}")

Task 1 Verification:
SendTime invalid formats: 0
SendTime invalid dates: 0
OpenTime invalid formats: 0
OpenTime invalid dates: 0
ClickTime invalid formats: 0
ClickTime invalid dates: 0


## Task 2: Fix RecipientEmail
Validate `RecipientEmail` to ensure valid email formats (e.g., 'user@domain.com'), removing invalid entries (e.g., 'user@', 'n/a'). Approximately 5% of ~1,020 rows are invalid. Valid emails must have a username, '@', domain, and top-level domain (e.g., '.com'). Dropping invalid emails ensures accurate campaign tracking (e.g., open rates by recipient).

In [81]:
# Task 2: Fix RecipientEmail

# Preserve original RecipientEmail for reference
df_clean['OriginalRecipientEmail'] = df_clean['RecipientEmail']

# Define valid email pattern
email_pattern = r'^[\w\.-]+@[\w\.-]+\.\w+$'

# Validate emails and filter out invalid ones
valid_emails = df_clean['RecipientEmail'].str.match(email_pattern, na=False)
df_clean = df_clean[valid_emails].copy() # Ensure fresh copy and avoid warnings 

# Log rows dropped
print(f"Rows before Task 2: {len(df_clean) + (~valid_emails).sum()}")
print(f"Rows after Task 2: {len(df_clean)}")
print(f"Invalid emails removed: {(~valid_emails).sum()}")

Rows before Task 2: 1020
Rows after Task 2: 979
Invalid emails removed: 41


In [82]:
# Verify Task 2
print("Task 2 Verification:")
# Check for remaining invalid emails
invalid_emails = ~df_clean['RecipientEmail'].str.match(email_pattern, na=False)
print(f"Remaining invalid emails: {invalid_emails.sum()}")
if invalid_emails.any():
    print(f"Invalid examples:\n{df_clean[invalid_emails][['RecipientEmail']].head()}")

# Sample valid emails to confirm
print("\nSample of valid emails:")
print(df_clean['RecipientEmail'].sample(5, random_state=42))

# Check for unexpected NaN
print(f"NaN in RecipientEmail: {df_clean['RecipientEmail'].isna().sum()}")

Task 2 Verification:
Remaining invalid emails: 0

Sample of valid emails:
209      user186@gmail.com
566       user83@yahoo.com
182     user46@company.com
967    user117@company.com
67       user134@gmail.com
Name: RecipientEmail, dtype: object
NaN in RecipientEmail: 0


## Task 3: Standardize CampaignID
Normalize `CampaignID` to uppercase (e.g., 'camp101' → 'CAMP101') to ensure consistent campaign tracking. Approximately 5% of ~969 rows (post-Task 2) have lowercase or mixed-case IDs (e.g., 'camp101', 'Camp102'). Valid IDs follow the pattern 'CAMP' + 3 digits. Standardization enables accurate grouping for metrics like open rates by campaign.

In [84]:
# Task 3: Standardize CampaignID

# Preserve original CampaignID for reference
df_clean['OriginalCampaignID'] = df_clean['CampaignID']

# Normalize to uppercase
df_clean['CampaignID'] = df_clean['CampaignID'].str.upper()

# Log changes
print(f"Rows before Task 3: {len(df_clean)}")
print(f"Unique CampaignIDs before: {df_clean['OriginalCampaignID'].nunique()}")
print(f"Unique CampaignIDs after: {df_clean['CampaignID'].nunique()}")

Rows before Task 3: 979
Unique CampaignIDs before: 6
Unique CampaignIDs after: 3


In [85]:
# Verify Task 3
print("Task 3 Verification:")
# Check for non-uppercase or invalid IDs
id_pattern = r'^CAMP\d{3}$'
valid_ids = df_clean['CampaignID'].str.match(id_pattern, na=False)
print(f"Invalid CampaignIDs: {(~valid_ids).sum()}")
if (~valid_ids).any():
    print(f"Invalid examples:\n{df_clean[~valid_ids][['CampaignID']].head()}")

# Sample CampaignIDs to confirm
print("\nSample of standardized CampaignIDs:")
print(df_clean['CampaignID'].sample(5, random_state=42))

# Check for NaN
print(f"NaN in CampaignID: {df_clean['CampaignID'].isna().sum()}")

# Confirm unique values
print(f"Unique CampaignIDs: {df_clean['CampaignID'].value_counts()}")

Task 3 Verification:
Invalid CampaignIDs: 0

Sample of standardized CampaignIDs:
209    CAMP101
566    CAMP102
182    CAMP103
967    CAMP102
67     CAMP101
Name: CampaignID, dtype: object
NaN in CampaignID: 0
Unique CampaignIDs: CampaignID
CAMP103    335
CAMP101    333
CAMP102    311
Name: count, dtype: int64


## Task 4: Clean Conversion
Normalize `Conversion` to 'Yes'/'No', fixing inconsistent entries (e.g., 'Y', 'N') and filling blanks (''). Approximately 10% of ~969 rows (post-Task 3) have invalid entries ('Y', 'N', or ''), with ~90% as 'Yes'/'No'. Standardization ensures accurate conversion rate analysis for email campaigns.

In [96]:
# Task 4: Clean Conversion

# Preserve original Conversion for reference
df_clean['OriginalConversion'] = df_clean['Conversion']

# Define mapping for normalization
conversion_map = {
    'Yes': 'Yes',
    'Y': 'Yes',
    'No': 'No',
    'N': 'No',
    '': 'No'  # Blanks imply no conversion
}

# Normalize Conversion
df_clean['Conversion'] = df_clean['Conversion'].replace(conversion_map)

# Log changes
print(f"Rows before Task 4: {len(df_clean)}")
print(f"Original Conversion values:\n{df_clean['OriginalConversion'].value_counts(dropna=False)}")

Rows before Task 4: 979
Original Conversion values:
OriginalConversion
Yes    246
Y      230
NaN    210
No     204
N       89
Name: count, dtype: int64


In [98]:
# Verify Task 4
print("Task 4 Verification:")
# Check for valid values (only 'Yes' or 'No')
valid_conversions = df_clean['Conversion'].isin(['Yes', 'No'])
print(f"Invalid Conversions: {(~valid_conversions).sum()}")
if (~valid_conversions).any():
    print(f"Invalid examples:\n{df_clean[~valid_conversions][['Conversion']].head()}")

# Sample Conversion values
print("\nSample of standardized Conversions:")
print(df_clean['Conversion'].sample(5, random_state=42))

# Check for NaN
print(f"NaN in Conversion: {df_clean['Conversion'].isna().sum()}")

# Show final value counts
print(f"Final Conversion values:\n{df_clean['Conversion'].value_counts()}")

Task 4 Verification:
Invalid Conversions: 210
Invalid examples:
   Conversion
0         NaN
3         NaN
15        NaN
26        NaN
30        NaN

Sample of standardized Conversions:
209    Yes
566    Yes
182    NaN
967    NaN
67     Yes
Name: Conversion, dtype: object
NaN in Conversion: 210
Final Conversion values:
Conversion
Yes    476
No     293
Name: count, dtype: int64


## Task 5: Resolve Duplicates
Remove exact duplicate `EmailID` entries and flag near-duplicates (same `RecipientEmail` and `CampaignID` within 5 minutes) in ~969 rows (post-Task 4). Approximately 2% of rows (~20) are duplicates due to logging errors. Deduplication ensures accurate campaign metrics (e.g., unique email sends).

In [105]:
# Task 5: Resolve Duplicates

# Preserve original DataFrame indices
df_clean['OriginalIndex'] = df_clean.index

# Drop exact EmailID duplicates
df_clean = df_clean.drop_duplicates(subset=['EmailID'], keep='first').copy()

# Sort by RecipientEmail, CampaignID, SendTime for near-duplicate check
df_clean = df_clean.sort_values(['RecipientEmail', 'CampaignID', 'SendTime'])

# Flag near-duplicates (same RecipientEmail, CampaignID within 5 minutes)
df_clean['SendTime_dt'] = pd.to_datetime(df_clean['SendTime'], errors='coerce')
time_diff = df_clean['SendTime_dt'].diff().abs() <= pd.Timedelta(minutes=5)
near_dups = df_clean.duplicated(subset=['RecipientEmail', 'CampaignID'], keep=False) & time_diff
near_dups = near_dups.fillna(False)

# Remove near-duplicates and reset index
df_clean = df_clean[~near_dups].copy().reset_index(drop=True)

# Log changes
print(f"Rows before Task 5: {len(df_clean) + near_dups.sum() + df_clean['EmailID'].duplicated().sum()}")
print(f"Exact EmailID duplicates removed: {df_clean['EmailID'].duplicated().sum()}")
print(f"Near-duplicates removed: {near_dups.sum()}")
print(f"Rows after Task 5: {len(df_clean)}")

Rows before Task 5: 959
Exact EmailID duplicates removed: 0
Near-duplicates removed: 0
Rows after Task 5: 959


In [109]:
# Verify Task 5
print("Task 5 Verification:")
# Check index continuity
print(f"Index is sequential: {df_clean.index.equals(pd.RangeIndex(start=0, stop=len(df_clean)))}")

# Check for exact EmailID duplicates
print(f"Remaining EmailID duplicates: {df_clean['EmailID'].duplicated().sum()}")

# Sample data to confirm
print("\nSample of cleaned data:")
print(df_clean[['EmailID', 'RecipientEmail', 'CampaignID', 'SendTime']].sample(5, random_state=42))

# Check for NaN in key columns
print(f"NaN in EmailID: {df_clean['EmailID'].isna().sum()}")
print(f"NaN in RecipientEmail: {df_clean['RecipientEmail'].isna().sum()}")
print(f"NaN in SendTime: {df_clean['SendTime'].isna().sum()}")

# Drop temporary SendTime_dt, keep OriginalIndex for now
df_clean = df_clean.drop(columns=['SendTime_dt'])

Task 5 Verification:
Index is sequential: True
Remaining EmailID duplicates: 0

Sample of cleaned data:
       EmailID      RecipientEmail CampaignID             SendTime
836  EMAIL1017  user74@outlook.com    CAMP102  2024-01-17 07:11:43
477  EMAIL1371   user191@gmail.com    CAMP102  2024-01-22 13:00:10
350  EMAIL1555  user16@company.com    CAMP103  2024-01-19 21:46:07
892  EMAIL1554  user86@outlook.com    CAMP102  2024-01-25 05:17:14
855  EMAIL1245  user78@outlook.com    CAMP102  2024-01-13 13:27:33
NaN in EmailID: 0
NaN in RecipientEmail: 0
NaN in SendTime: 0


In [111]:
df_clean

,EmailID,RecipientEmail,CampaignID,SendTime,OpenTime,ClickTime,Conversion,Unsubscribe,OriginalSendTime,OriginalOpenTime,OriginalClickTime,OriginalRecipientEmail,OriginalCampaignID,OriginalConversion,OriginalIndex
0,EMAIL1751,user100@company.com,CAMP101,2024-01-30 13:07:52,2024-01-30 13:07:52,2024-01-30 13:07:52,NaN,Yes,2024-01-30 13:07:52,2024-01-30 13:07:52,2024-01-30 13:07:52,user100@company.com,CAMP101,NaN,750
1,EMAIL1641,user100@company.com,CAMP103,2024-01-12 01:02:04,2024-01-12 01:02:04,2024-01-12 01:02:04,Yes,No,01/12/2024 01:02:04,01/12/2024 01:02:04,01/12/2024 01:02:04,user100@company.com,CAMP103,Y,640
2,EMAIL1692,user100@gmail.com,CAMP103,2024-01-02 09:07:47,NaN,NaN,Yes,Yes,2024-01-02 09:07:47,NaN,NaN,user100@gmail.com,CAMP103,Yes,691
3,EMAIL1137,user100@outlook.com,CAMP102,2024-01-23 21:27:39,NaN,NaN,No,No,01/23/2024 21:27:39,NaN,NaN,user100@outlook.com,CAMP102,N,136
4,EMAIL1130,user100@yahoo.com,CAMP102,2024-01-29 04:42:28,NaN,NaN,Yes,Yes,01/29/2024 04:42:28,NaN,NaN,user100@yahoo.com,CAMP102,Y,129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
954,EMAIL1953,user99@outlook.com,CAMP103,2024-01-06 16:46:42,2024-01-06 16:46:42,2024-01-06 16:46:42,Yes,Yes,01/06/2024 16:46:42,01/06/2024 16:46:42,01/06/2024 16:46:42,user99@outlook.com,CAMP103,Y,952
955,EMAIL1679,user9@company.com,CAMP103,2024-01-21 16:26:59,NaN,NaN,Yes,No,2024-01-21 16:26:59,NaN,NaN,user9@company.com,CAMP103,Y,678
956,EMAIL1485,user9@gmail.com,CAMP101,2024-01-14 04:14:50,NaN,NaN,No,Yes,01/14/2024 04:14:50,NaN,NaN,user9@gmail.com,CAMP101,No,484
957,EMAIL1745,user9@gmail.com,CAMP101,2024-01-22 15:49:26,2024-01-22 15:49:26,NaN,No,No,2024-01-22 15:49:26,2024-01-22 15:49:26,NaN,user9@gmail.com,CAMP101,N,744


## Task 6: Fill Blanks
Normalize `Unsubscribe` to 'Yes'/'No', fixing inconsistent entries (e.g., 'Y', 'N') and any unexpected blanks in ~969 rows (post-Task 5). Approximately 5% of rows (~48) may have inconsistent values. `OpenTime` and `ClickTime` NaN (~30%) are preserved as valid (no action needed, verified in Task 1). Normalization ensures accurate unsubscribe rate analysis.

In [115]:
# Task 6: Fill Blanks

# Preserve original Unsubscribe
df_clean['OriginalUnsubscribe'] = df_clean['Unsubscribe']

# Normalize Unsubscribe to 'Yes'/'No'
unsubscribe_map = {
    'Yes': 'Yes',
    'Y': 'Yes',
    'No': 'No',
    'N': 'No',
    '': 'No'  # Handle unexpected blanks
}
df_clean['Unsubscribe'] = df_clean['Unsubscribe'].replace(unsubscribe_map)

# Log changes
print(f"Rows before Task 6: {len(df_clean)}")
print(f"Original Unsubscribe values:\n{df_clean['OriginalUnsubscribe'].value_counts(dropna=False)}")

Rows before Task 6: 959
Original Unsubscribe values:
OriginalUnsubscribe
No     493
Yes    466
Name: count, dtype: int64


In [117]:
# Verify Task 6
print("Task 6 Verification:")
# Check Unsubscribe values
valid_unsub = df_clean['Unsubscribe'].isin(['Yes', 'No'])
print(f"Invalid Unsubscribe values: {(~valid_unsub).sum()}")
if (~valid_unsub).any():
    print(f"Invalid examples:\n{df_clean[~valid_unsub][['Unsubscribe']].head()}")

# Sample data
print("\nSample of cleaned data:")
print(df_clean[['RecipientEmail', 'Unsubscribe']].sample(5, random_state=42))

# Check for NaN
print(f"NaN in Unsubscribe: {df_clean['Unsubscribe'].isna().sum()}")

# Final Unsubscribe counts
print(f"Final Unsubscribe values:\n{df_clean['Unsubscribe'].value_counts()}")

Task 6 Verification:
Invalid Unsubscribe values: 0

Sample of cleaned data:
         RecipientEmail Unsubscribe
836  user74@outlook.com          No
477   user191@gmail.com         Yes
350  user16@company.com          No
892  user86@outlook.com         Yes
855  user78@outlook.com         Yes
NaN in Unsubscribe: 0
Final Unsubscribe values:
Unsubscribe
No     493
Yes    466
Name: count, dtype: int64


In [119]:
# Create final cleaned DataFrame
cleaned_cols = ['EmailID', 'RecipientEmail', 'CampaignID', 'SendTime', 'OpenTime', 
                'ClickTime', 'Conversion', 'Unsubscribe']
df_final = df_clean[cleaned_cols].copy()

# Save final dataset
df_final.to_csv('cleaned_email_campaign.csv', index=False)

# Log final state
print("Final Cleaned DataFrame:")
print(df_final.info())
print("\nSample of final data:")
print(df_final.sample(5, random_state=42))

Final Cleaned DataFrame:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 959 entries, 0 to 958
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   EmailID         959 non-null    object
 1   RecipientEmail  959 non-null    object
 2   CampaignID      959 non-null    object
 3   SendTime        959 non-null    object
 4   OpenTime        285 non-null    object
 5   ClickTime       152 non-null    object
 6   Conversion      753 non-null    object
 7   Unsubscribe     959 non-null    object
dtypes: object(8)
memory usage: 60.1+ KB
None

Sample of final data:
       EmailID      RecipientEmail CampaignID             SendTime  \
836  EMAIL1017  user74@outlook.com    CAMP102  2024-01-17 07:11:43   
477  EMAIL1371   user191@gmail.com    CAMP102  2024-01-22 13:00:10   
350  EMAIL1555  user16@company.com    CAMP103  2024-01-19 21:46:07   
892  EMAIL1554  user86@outlook.com    CAMP102  2024-01-25 05:17:14   
855  EMAIL12